# Independent-cohort gene signatures — D3

Cohort-specific gene selection using logistic LASSO and Elastic Net. Regularisation is tuned exclusively within this cohort by stratified cross-validation. Half-sample selection frequencies describe exploratory within-cohort robustness; they are not the published Meinshausen–Bühlmann Stability Selection method and imply no formal error-control guarantee.

Inputs: accepted sample-aligned RNA-seq dataset. Outputs: complete coefficients, selected genes, CV scores, subsample assignments, selection frequencies, reports, and checksums.


In [1]:
%%writefile /kaggle/working/full_gene_loco.py
"""Independent-cohort full-gene LASSO / Elastic Net nested evaluation."""
from __future__ import annotations

import csv
import hashlib
import io
import json
import os
import platform
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd
import scipy
import sklearn
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, balanced_accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

EXPECTED_ZIP_SHA = '8a83f20035fcac5ae5d8bfe496e4d7491e47400b61ca1080b828c8ad54e91a74'
EXPECTED_CSV_SHA = {
    'count_d1_aligned.csv': '0f595746ea572a349580d9d3c0242559345b88c927c161b1982933102e6c812d',
    'count_d2_aligned.csv': '2ff413b813e3e4d10927da7f8342eccb35a00205c319c0eb714388aea97f245d',
    'count_d3_aligned.csv': '33b2e51e682da4c41f53becf00b1fc88bb5710293ce2d9d87837bca86a7b3d48',
    'count_d4_aligned.csv': 'c26d8db65b335c89967bd4e072c61a6b7adcb57715c5fe36107d9bd8cd9409e0',
    'desc_d1.csv': '91aadec65bb7f2c1945e7941222b9f490a0c9041d495b8d22606f614484c7b33',
    'desc_d2.csv': '4198f574167754eb00e007ded8cfc2528d50ead0d80b96c1a5cc8139131a140b',
    'desc_d3.csv': '9db3c39975a50afcbd851d3cdcdf56f1e59e7f9bbe18b3b80fd7328400a31332',
    'desc_d4.csv': 'bdd5f6d5ae9a6713470f1eadab2e62a961a38d417f27401359d55b5d906a1193',
}
COHORTS = ('d1','d2','d3','d4')
C_GRID = (0.001, 0.01, 0.1, 1.0)
ELASTIC_RATIOS = (0.5, 0.9)
MIN_CPM = 1.0
MIN_FRACTION = 0.1
SEED = 355
MAX_ITER = 1600
TOL = 0.001


def sha_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def require(condition: bool, reason: str) -> None:
    if not condition:
        raise ValueError(reason)


def find_input(directory: Path):
    root = Path(directory)
    require(root.exists(), f'Input root not found: {root}')
    unpacked = [p for p in root.rglob('count_d1_aligned.csv') if p.is_file()]
    for file in unpacked:
        folder = file.parent
        if all((folder / name).exists() for name in EXPECTED_CSV_SHA):
            return ('directory', folder)
    archives = [p for p in root.rglob('ibd_dataset_sample_aligned.zip') if p.is_file()]
    require(len(archives) == 1, 'Attach exactly one accepted dataset input containing the eight aligned CSV files')
    return ('archive', archives[0])


def load_data(directory: Path) -> tuple[pd.DataFrame, np.ndarray, tuple[str,...], dict]:
    mode, source = find_input(directory)
    if mode == 'archive':
        require(sha_bytes(source.read_bytes()) == EXPECTED_ZIP_SHA, 'Dataset ZIP checksum mismatch')
        with ZipFile(source) as zf:
            require(zf.testzip() is None, 'ZIP CRC verification failed')
            raw = {name: zf.read(name) for name in EXPECTED_CSV_SHA}
    else:
        raw = {name: (source / name).read_bytes() for name in EXPECTED_CSV_SHA}
    for name, expected in EXPECTED_CSV_SHA.items():
        require(sha_bytes(raw[name]) == expected, f'Dataset file checksum mismatch: {name}')
    headers = {}
    for cohort in COHORTS:
        names = next(csv.reader(io.StringIO(raw[f'count_{cohort}_aligned.csv'].decode('utf-8-sig'))))
        require(len(names) == len(set(names)) and names[0] == 'ID', f'Invalid gene names: {cohort}')
        headers[cohort] = names
    genes = tuple(sorted(set.intersection(*(set(headers[c][1:]) for c in COHORTS))))
    require(len(genes) == 9983, 'Unexpected shared gene count')
    counts = []
    metadata = []
    expected_rows = dict(zip(COHORTS, (260,179,76,174)))
    for cohort in COHORTS:
        desc = pd.read_csv(io.BytesIO(raw[f'desc_{cohort}.csv']), dtype={'ID':'string','Study':'string','class':'string'})
        df = pd.read_csv(io.BytesIO(raw[f'count_{cohort}_aligned.csv']), usecols=['ID',*genes], dtype={'ID':'string'})
        require(len(desc)==len(df)==expected_rows[cohort], f'{cohort}: incorrect sample count')
        require(desc['ID'].tolist()==df['ID'].tolist(), f'{cohort}: sample alignment mismatch')
        require({'ID','Study','class'} <= set(desc.columns), f'{cohort}: missing clinical fields')
        require(set(desc['class'])=={'CD','Not IBD'}, f'{cohort}: unexpected classes')
        mat = df.loc[:, list(genes)].to_numpy(dtype=np.float64, copy=True)
        require(np.isfinite(mat).all() and (mat>=0).all() and (mat.sum(axis=1)>0).all(), f'{cohort}: invalid counts')
        desc = desc[['ID','Study','class']].copy()
        desc.insert(0,'cohort',cohort)
        metadata.append(desc)
        counts.append(mat)
    meta = pd.concat(metadata, ignore_index=True)
    require(meta['ID'].is_unique and len(meta)==689, 'Duplicate or missing patients')
    require(meta['Study'].nunique()==6, 'BioProject count mismatch')
    return meta, np.vstack(counts), genes, {'source_mode':mode,'source_path':str(source), 'zip_sha256':EXPECTED_ZIP_SHA,'csv_sha256':EXPECTED_CSV_SHA}


def log_cpm(values: np.ndarray) -> np.ndarray:
    depths = values.sum(axis=1)
    require(np.all(depths>0), 'Empty library detected')
    return np.log2(1.0 + 1e6 * values / depths[:, None])


def fit_features(train_counts: np.ndarray) -> tuple[np.ndarray, np.ndarray, StandardScaler, dict]:
    x = log_cpm(train_counts)
    threshold = max(2, int(np.ceil(MIN_FRACTION * len(x))))
    valid = ((x > np.log2(1+MIN_CPM)).sum(axis=0) >= threshold) & (x.var(axis=0) > 1e-12)
    indices = np.flatnonzero(valid)
    require(len(indices) >= 2, 'Insufficient genes after training-only filtering')
    scaler = StandardScaler()
    fitted = scaler.fit_transform(x[:, indices])
    require(np.isfinite(fitted).all(), 'Invalid prepared features')
    return fitted, indices, scaler, {'fit_patients':len(x), 'prevalence_threshold_samples':threshold,'eligible_genes':len(indices)}


def apply_features(counts: np.ndarray, indices: np.ndarray, scaler: StandardScaler) -> np.ndarray:
    return scaler.transform(log_cpm(counts)[:, indices])


def model_fit(x: np.ndarray, y: np.ndarray, C: float, ratio: float):
    sklearn_major_minor = tuple(int(z) for z in sklearn.__version__.split('.')[:2])
    args = {'l1_ratio':float(ratio)} if sklearn_major_minor >= (1,8) else {
        'penalty':'l1' if ratio==1 else 'elasticnet',
        'l1_ratio': None if ratio==1 else float(ratio)}
    model = LogisticRegression(solver='saga',C=float(C),class_weight='balanced',
                               max_iter=MAX_ITER,tol=TOL,random_state=SEED,**args)
    with warnings.catch_warnings(record=True) as recorded:
        warnings.simplefilter('always', ConvergenceWarning)
        model.fit(x,y)
    converged = not any(issubclass(r.category,ConvergenceWarning) for r in recorded)
    return model, converged


def metrics(y: np.ndarray, prob: np.ndarray) -> dict:
    require(np.isfinite(prob).all() and ((0<=prob)&(prob<=1)).all(), 'Invalid probabilities')
    pred = (prob >= .5).astype(int)
    tn, fp, fn, tp = [int(v) for v in confusion_matrix(y,pred,labels=[0,1]).ravel()]
    return dict(auroc=float(roc_auc_score(y,prob)), average_precision=float(average_precision_score(y,prob)),
                balanced_accuracy=float(balanced_accuracy_score(y,pred)),
                sensitivity=tp/(tp+fn), specificity=tn/(tn+fp),
                tp=tp,tn=tn,fp=fp,fn=fn)


def safe_csv(df: pd.DataFrame, path: Path):
    temporary = path.with_suffix(path.suffix+'.tmp')
    df.to_csv(temporary, index=False)
    os.replace(temporary,path)


def safe_json(obj: dict, path: Path):
    temporary = path.with_suffix('.tmp')
    temporary.write_text(json.dumps(obj,indent=2,sort_keys=True),encoding='utf-8')
    os.replace(temporary,path)


def run(heldout: str, input_root: Path, output_root: Path) -> Path:
    require(heldout in COHORTS, 'Unknown outer cohort')
    started = time.monotonic()
    meta,counts,genes,source=load_data(input_root)
    y = (meta['class']=='CD').to_numpy(dtype=int)
    cohorts=meta['cohort'].to_numpy()
    test_ids=np.flatnonzero(cohorts==heldout)
    train_ids=np.flatnonzero(cohorts!=heldout)
    require(len(test_ids)>0 and len(train_ids)+len(test_ids)==689, 'Outer fold coverage invalid')
    target=Path(output_root) / heldout
    require(not (target/'report.json').exists(), 'Existing completed results cannot be overwritten')
    target.mkdir(parents=True,exist_ok=True)
    scores=[]
    selected=[]
    predictions=[]
    methods={ 'lasso':(1.0,), 'elastic_net':ELASTIC_RATIOS }
    output={ 'status':'RUNNING', 'heldout':heldout,'dataset':source, 'config':{
        'folds':'nested leave-one-analytical-cohort-out', 'gene_universe':'9983 genes shared before inspecting labels',
        'normalization':'log2(1 + CPM) per sample on shared gene universe',
        'filter':'training-only CPM > 1 in >=10% of training samples, nonzero variance',
        'scaling':'training-only standardization','outer_score_threshold':0.5,
        'tuning_metric':'mean of 3 independent cohort-validation AUROCs',
        'C_grid':C_GRID, 'elastic_ratios':ELASTIC_RATIOS,'max_iter':MAX_ITER,'tol':TOL,
        'class_weight':'balanced','random_seed':SEED}, 'software':{
        'python':platform.python_version(), 'numpy':np.__version__,'pandas':pd.__version__,
        'scipy':scipy.__version__,'sklearn':sklearn.__version__},
        'candidates':[], 'results':[], 'start_utc':datetime.now(timezone.utc).isoformat() }
    safe_json(output,target/'progress.json')
    for method, ratios in methods.items():
        configs=[(C,r) for r in ratios for C in C_GRID]
        candidate_scores={ (float(c),float(r)):[] for c,r in configs }
        for inner in COHORTS:
            if inner == heldout:
                continue
            ix=np.flatnonzero((cohorts!=heldout)&(cohorts!=inner))
            vx=np.flatnonzero(cohorts==inner)
            require(np.intersect1d(np.r_[ix,vx],test_ids).size==0, 'Outer test leakage')
            require(np.intersect1d(ix,vx).size==0, 'Inner sample overlap')
            train_matrix, inds, scaler, prep=fit_features(counts[ix])
            val_matrix=apply_features(counts[vx],inds,scaler)
            print(f'{heldout}/{method}: validating on {inner} using {len(inds)} genes', flush=True)
            for c,r in configs:
                model, conv = model_fit(train_matrix,y[ix],c,r)
                prob=model.predict_proba(val_matrix)[:,1]
                score=metrics(y[vx],prob)
                scores.append(dict(outer=heldout,method=method,validation=inner,C=c,l1_ratio=r,
                                   n_fit=len(ix),n_validation=len(vx),
                                   n_candidate_genes=len(inds), nonzero=int((np.abs(model.coef_[0])>1e-8).sum()),
                                   converged=conv, n_iter=int(model.n_iter_[0]),**score))
                candidate_scores[(float(c),float(r))].append(score['auroc'])
        require(all(len(z)==3 for z in candidate_scores.values()), 'Incomplete inner evaluation')
        ranked=sorted(candidate_scores,key=lambda cr:(-np.mean(candidate_scores[cr]),cr[0],-cr[1]))
        c,r=ranked[0]
        all_x, inds,scaler,preprocess=fit_features(counts[train_ids])
        heldout_x=apply_features(counts[test_ids],inds,scaler)
        final,conv=model_fit(all_x,y[train_ids],c,r)
        pr=final.predict_proba(heldout_x)[:,1]
        result=dict(method=method,outer=heldout,C=c,l1_ratio=r,
                    inner_mean_auroc=float(np.mean(candidate_scores[(c,r)])),
                    train_patients=len(train_ids),test_patients=len(test_ids),
                    candidate_gene_count=len(inds),nonzero_coefficients=int((np.abs(final.coef_[0])>1e-8).sum()),
                    converged=conv,n_iter=int(final.n_iter_[0]),preprocessing=preprocess,**metrics(y[test_ids],pr))
        output['results'].append(result)
        for i,prob in zip(test_ids,pr):
            predictions.append(dict(outer=heldout,method=method,ID=str(meta.iloc[i]['ID']),
                                    Study=str(meta.iloc[i]['Study']),class_true=str(meta.iloc[i]['class']),
                                    y_true=int(y[i]),probability_CD=float(prob)))
        for position in np.flatnonzero(np.abs(final.coef_[0])>1e-8):
            selected.append(dict(outer=heldout,method=method,gene=genes[inds[position]],
                                 coef_scaled=float(final.coef_[0,position])))
        output['candidates'].extend([dict(method=method,C=z[0],l1_ratio=z[1],
                               mean_validation_auroc=float(np.mean(candidate_scores[z]))) for z in ranked])
        safe_csv(pd.DataFrame(scores),target/'inner_scores.csv')
        safe_csv(pd.DataFrame(predictions),target/'predictions.csv')
        safe_csv(pd.DataFrame(selected,columns=['outer','method','gene','coef_scaled']),target/'selected_genes.csv')
        safe_json(output,target/'progress.json')
        print(f"{heldout}/{method}: AUROC={result['auroc']:.3f}, selected={result['nonzero_coefficients']}, converged={conv}",flush=True)
    require(len(output['results'])==2 and len(predictions)==2*len(test_ids),'Incomplete outer fold')
    require(all(len({x['ID'] for x in predictions if x['method']==m})==len(test_ids) for m in methods),
            'Duplicate outer predictions')
    output['status']='PASS' if all(item['converged'] for item in output['results']) and all(s['converged'] for s in scores) else 'WARN_CONVERGENCE'
    output['elapsed_seconds']=round(time.monotonic()-started,2)
    output['completed_utc']=datetime.now(timezone.utc).isoformat()
    output['artifacts_sha256']={f:hashlib.sha256((target/f).read_bytes()).hexdigest()
                                for f in ('inner_scores.csv','predictions.csv','selected_genes.csv')}
    safe_json(output,target/'report.json')
    print(f'FINAL_STATUS={output["status"]}; HELDOUT={heldout}; ELAPSED_SECONDS={output["elapsed_seconds"]}',flush=True)
    return target


Writing /kaggle/working/full_gene_loco.py


In [2]:
%%writefile /kaggle/working/independent_signatures.py
"""Independent-cohort signatures and exploratory subsampling robustness."""
from __future__ import annotations

import hashlib
import json
import platform
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

COHORTS = ('d1', 'd2', 'd3', 'd4')
C_GRID = (0.001, 0.01, 0.1, 1.0)
METHOD_RATIOS = {'lasso': (1.0,), 'elastic_net': (0.5, 0.9)}
SEED = 355
RESAMPLES = 30
COEF_CUTOFF = 1e-8


def require_true(condition, message):
    if not condition:
        raise ValueError(message)


def stratified_half_sample(labels: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    parts = []
    for klass in (0, 1):
        locations = np.flatnonzero(labels == klass)
        require_true(len(locations) >= 4, f'Insufficient observations in class {klass}')
        take = len(locations) // 2
        parts.append(rng.choice(locations, size=take, replace=False))
    ids = np.sort(np.concatenate(parts))
    require_true(len(ids) == len(set(ids.tolist())), 'Non-unique subsample')
    require_true(set(labels[ids]) == {0, 1}, 'Missing subsample class')
    return ids


def fit_cohort(meta: pd.DataFrame, counts: np.ndarray, genes: tuple[str, ...],
               cohort: str, target_root: Path, *, n_resamples: int = RESAMPLES,
               c_grid: tuple[float, ...] = C_GRID,
               ratio_grid: dict | None = None, cv_folds: int = 3) -> Path:
    from full_gene_loco import fit_features, model_fit, safe_csv, safe_json

    require_true(cohort in COHORTS, 'Unknown cohort')
    require_true(n_resamples >= 1 and cv_folds >= 2, 'Invalid experiment configuration')
    require_true(len(meta) == len(counts) and counts.shape[1] == len(genes), 'Data shape mismatch')
    require_true(meta['ID'].is_unique, 'Nonunique sample identifiers')
    ratio_grid = ratio_grid if ratio_grid is not None else METHOD_RATIOS
    sample_ids = np.flatnonzero(meta['cohort'].to_numpy() == cohort)
    y_all = (meta['class'].to_numpy() == 'CD').astype(int)
    y = y_all[sample_ids]
    require_true(set(y) == {0, 1}, 'Both classes required')
    require_true(np.bincount(y).min() >= cv_folds, 'Insufficient samples for CV')
    x = counts[sample_ids]
    identifiers = meta.iloc[sample_ids]['ID'].astype(str).to_numpy()
    output = Path(target_root) / cohort
    require_true(not (output / 'report.json').exists(), 'Completed output cannot be overwritten')
    output.mkdir(parents=True, exist_ok=True)
    started = time.monotonic()
    summary = {'status': 'RUNNING', 'cohort': cohort, 'n_samples': int(len(x)),
               'n_cases': int(sum(y)), 'n_controls': int(len(y) - sum(y)),
               'n_shared_genes': len(genes), 'software': {
                   'python': platform.python_version(), 'numpy': np.__version__,
                   'pandas': pd.__version__, 'scipy': scipy.__version__,
                   'sklearn': sklearn.__version__},
               'configuration': {'seed': SEED, 'cv_folds': cv_folds,
                                 'cv_metric': 'ROC_AUC', 'c_grid': list(c_grid),
                                 'method_ratios': {k: list(v) for k, v in ratio_grid.items()},
                                 'resamples': n_resamples,
                                 'resample_type': 'stratified_without_replacement_half_sample',
                                 'preprocessing': 'training-only log2(1+CPM), CPM>1 in >=10%, standardized',
                                 'class_weight': 'balanced', 'coefficient_cutoff': COEF_CUTOFF},
               'completed_methods': [], 'created_utc': datetime.now(timezone.utc).isoformat()}
    safe_json(summary, output / 'progress.json')
    cv_records, fits, selections, frequencies, subsamples, resample_summary, all_coeffs = [], [], [], [], [], [], []
    config_list = [(m, float(c), float(r)) for m, ratios in ratio_grid.items()
                   for r in ratios for c in c_grid]
    cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=SEED + 47)
    cv_splits = list(cv.split(x, y))
    for split_number, (train, valid) in enumerate(cv_splits, 1):
        transformed, col_idx, scaler, _ = fit_features(x[train])
        from full_gene_loco import apply_features
        held = apply_features(x[valid], col_idx, scaler)
        for method, c, ratio in config_list:
            fit, converged = model_fit(transformed, y[train], c, ratio)
            probabilities = fit.predict_proba(held)[:, 1]
            score = float(roc_auc_score(y[valid], probabilities))
            cv_records.append({'cohort': cohort, 'method': method, 'fold': split_number,
                               'C': c, 'l1_ratio': ratio, 'auroc': score,
                               'n_train': len(train), 'n_validation': len(valid),
                               'n_input_genes': len(col_idx), 'converged': bool(converged),
                               'n_iter': int(fit.n_iter_[0])})
        safe_csv(pd.DataFrame(cv_records), output / 'cv_scores.csv')
    scores = pd.DataFrame(cv_records)
    final_params = {}
    for method, ratios in ratio_grid.items():
        method_scores = scores[scores['method'] == method].groupby(['C', 'l1_ratio'], as_index=False)['auroc'].mean()
        require_true(len(method_scores) == len(c_grid) * len(ratios), 'Incomplete tuning grid')
        method_scores = method_scores.sort_values(['auroc', 'C', 'l1_ratio'], ascending=[False, True, False])
        best = method_scores.iloc[0]
        c, ratio = float(best['C']), float(best['l1_ratio'])
        final_params[method] = (c, ratio)
        transformed, col_idx, scaler, prep = fit_features(x)
        fit, converged = model_fit(transformed, y, c, ratio)
        coefs = np.zeros(len(genes), dtype=np.float64)
        coefs[col_idx] = fit.coef_[0]
        support = np.flatnonzero(np.abs(coefs) > COEF_CUTOFF)
        fits.append({'cohort': cohort, 'method': method, 'C': c, 'l1_ratio': ratio,
                     'cv_mean_auroc': float(best['auroc']), 'n_selected': len(support),
                     'eligible_genes': len(col_idx), 'converged': bool(converged),
                     'n_iter': int(fit.n_iter_[0]), 'sample_n': len(x)})
        for position in support:
            selections.append({'cohort': cohort, 'method': method,
                               'gene': genes[position], 'coef_scaled': float(coefs[position]),
                               'abs_coef': float(abs(coefs[position]))})
        all_coeffs.extend({'cohort': cohort, 'method': method, 'gene': g, 'coef_scaled': float(coefs[j]),
                           'abs_coef': float(abs(coefs[j])), 'selected': bool(abs(coefs[j]) > COEF_CUTOFF)}
                          for j, g in enumerate(genes))
        safe_csv(pd.DataFrame(fits), output / 'signature_summary.csv')
        safe_csv(pd.DataFrame(selections, columns=['cohort','method','gene','coef_scaled','abs_coef']),
                 output / 'selected_genes.csv')
        safe_csv(pd.DataFrame(all_coeffs), output / 'all_gene_coefficients.csv')
        print(f'{cohort}/{method}: CV AUROC={best["auroc"]:.3f}; selected={len(support)}', flush=True)
    safe_json(summary | {'tuned_settings': final_params}, output / 'progress.json')
    for method, (c, ratio) in final_params.items():
        rng = np.random.default_rng(SEED + 1000 * COHORTS.index(cohort) + (0 if method == 'lasso' else 250))
        selection_counts = np.zeros(len(genes), dtype=np.int32)
        for iteration in range(1, n_resamples + 1):
            sampled = stratified_half_sample(y, rng)
            transformed, positions, scaler, prep = fit_features(x[sampled])
            fit, converged = model_fit(transformed, y[sampled], c, ratio)
            chosen = positions[np.flatnonzero(np.abs(fit.coef_[0]) > COEF_CUTOFF)]
            selection_counts[chosen] += 1
            resample_summary.append({'cohort': cohort, 'method': method, 'iteration': iteration,
                                     'n_samples': len(sampled), 'n_cases': int(y[sampled].sum()),
                                     'n_controls': int(len(sampled) - y[sampled].sum()),
                                     'n_eligible_genes': len(positions), 'n_selected': len(chosen),
                                     'converged': bool(converged), 'n_iter': int(fit.n_iter_[0])})
            subsamples.extend({'cohort': cohort, 'method': method, 'iteration': iteration,
                               'ID': identifiers[i], 'class_true': 'CD' if y[i] else 'Not IBD'}
                              for i in sampled)
            safe_csv(pd.DataFrame(resample_summary), output / 'subsample_fit_summary.csv')
            safe_csv(pd.DataFrame(subsamples), output / 'subsample_membership.csv')
            safe_json(summary | {'phase': 'resampling', 'method': method, 'iteration': iteration,
                                 'tuned_settings': final_params}, output / 'progress.json')
            if iteration % 5 == 0 or iteration == n_resamples:
                print(f'{cohort}/{method}: {iteration}/{n_resamples} half-samples', flush=True)
        frequencies.extend({'cohort': cohort, 'method': method, 'gene': gene,
                            'n_selected': int(selection_counts[j]),
                            'n_resamples': int(n_resamples),
                            'frequency': float(selection_counts[j] / n_resamples)}
                           for j, gene in enumerate(genes))
        safe_csv(pd.DataFrame(frequencies), output / 'subsample_gene_frequencies.csv')
        summary['completed_methods'].append(method)
        safe_json(summary, output / 'progress.json')
    file_list = ('cv_scores.csv', 'signature_summary.csv', 'selected_genes.csv',
                 'all_gene_coefficients.csv', 'subsample_fit_summary.csv',
                 'subsample_membership.csv', 'subsample_gene_frequencies.csv')
    for f in file_list:
        require_true((output / f).is_file() and (output / f).stat().st_size > 0, f'Missing output {f}')
    summary['status'] = ('PASS' if scores['converged'].all()
                         and all(x['converged'] for x in fits)
                         and all(x['converged'] for x in resample_summary)
                         else 'WARN_CONVERGENCE')
    summary['elapsed_seconds'] = round(time.monotonic() - started, 1)
    summary['final_parameters'] = {k: {'C': v[0], 'l1_ratio': v[1]} for k,v in final_params.items()}
    summary['final_models'] = fits
    summary['files_sha256'] = {f: hashlib.sha256((output/f).read_bytes()).hexdigest() for f in file_list}
    summary['completed_utc'] = datetime.now(timezone.utc).isoformat()
    safe_json(summary, output / 'report.json')
    print(f'FINAL_STATUS={summary["status"]}; COHORT={cohort}; output={output}', flush=True)
    return output


def run(cohort: str, input_root: Path, output_root: Path, *, n_resamples: int = RESAMPLES):
    from full_gene_loco import load_data
    meta, counts, genes, source = load_data(input_root)
    result = fit_cohort(meta, counts, genes, cohort, output_root, n_resamples=n_resamples)
    path = result / 'report.json'
    content = json.loads(path.read_text())
    content['dataset'] = source
    from full_gene_loco import safe_json
    safe_json(content, path)
    return result


Writing /kaggle/working/independent_signatures.py


In [3]:
import sys
import traceback
import json
from pathlib import Path
sys.path.insert(0, "/kaggle/working")
from independent_signatures import run
cohort = "d3"
root = Path("/kaggle/working/cmps355/independent_signatures")
try:
    output = run(cohort, Path("/kaggle/input"), root, n_resamples=30)
    report = json.loads((output / "report.json").read_text())
    print(json.dumps({"status": report["status"], "cohort": report["cohort"], "final_models": report["final_models"]}, indent=2))
except Exception as exc:
    failure = root / cohort / "failure.json"
    failure.parent.mkdir(parents=True, exist_ok=True)
    failure.write_text(json.dumps({"status":"FAIL", "error":str(exc), "traceback":traceback.format_exc()}, indent=2))
    raise


d3/lasso: CV AUROC=0.928; selected=1734
d3/elastic_net: CV AUROC=0.928; selected=1934
d3/lasso: 5/30 half-samples
d3/lasso: 10/30 half-samples
d3/lasso: 15/30 half-samples
d3/lasso: 20/30 half-samples
d3/lasso: 25/30 half-samples
d3/lasso: 30/30 half-samples
d3/elastic_net: 5/30 half-samples
d3/elastic_net: 10/30 half-samples
d3/elastic_net: 15/30 half-samples
d3/elastic_net: 20/30 half-samples
d3/elastic_net: 25/30 half-samples
d3/elastic_net: 30/30 half-samples
FINAL_STATUS=PASS; COHORT=d3; output=/kaggle/working/cmps355/independent_signatures/d3
{
  "status": "PASS",
  "cohort": "d3",
  "final_models": [
    {
      "C": 1.0,
      "cohort": "d3",
      "converged": true,
      "cv_mean_auroc": 0.9278499278499278,
      "eligible_genes": 8419,
      "l1_ratio": 1.0,
      "method": "lasso",
      "n_iter": 575,
      "n_selected": 1734,
      "sample_n": 76
    },
    {
      "C": 1.0,
      "cohort": "d3",
      "converged": true,
      "cv_mean_auroc": 0.9278499278499278,
      "e